In [1]:
%pip install -q torchscale==0.3.0 "transformers>=4.40,<5" datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 10.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.

In [9]:
from google.colab import drive
drive.mount("/content/drive")

import math
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer
from torchscale.architecture.config import RetNetConfig
from torchscale.architecture.retnet import RetNetDecoder

assert torch.cuda.is_available(), "Enable a GPU in Colab first."
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

SAVE_DIR = Path("/content/drive/MyDrive/AttentionLens_RetNet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "vocab_size": 50257,
    "embed_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout": 0.1,
}

SEQ_LEN = 128
BATCH_SIZE = 1             # Conservative starting point for GPU memory
GRAD_ACCUM = 8              # Smoke run; increase for longer training
LR = 3e-4
SAVE_EVERY = 100
RESUME = True
STEPS = 1000
VALIDATE_EVERY = 100
VALIDATION_BATCHES = 100
tokenizer = AutoTokenizer.from_pretrained("gpt2")
assert len(tokenizer) == CONFIG["vocab_size"]

print("Training tokens:", STEPS * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Training tokens: 1024000


In [10]:
class RetNetModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, embed_dim)

        config = RetNetConfig(
            decoder_embed_dim=embed_dim,
            decoder_value_embed_dim=embed_dim, # Explicitly align value projection size
            decoder_retention_heads=num_heads,
            decoder_ffn_embed_dim=embed_dim * 4,
            decoder_layers=num_layers,
            dropout=dropout,
            activation_dropout=dropout,
            no_scale_embedding=True,
            layernorm_embedding=False,
            vocab_size=vocab_size,
        )

        self.retnet = RetNetDecoder(
            config,
            embed_tokens=self.token_emb,
        )
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        out, _ = self.retnet(
            prev_output_tokens=x,
            features_only=True,
        )
        return self.fc_out(out)


model = RetNetModel(**CONFIG).to(DEVICE)

print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

# Check the forward pass before downloading/training on the dataset.
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    test_logits = model(
        torch.randint(0, CONFIG["vocab_size"], (1, SEQ_LEN), device=DEVICE)
    )

assert test_logits.shape == (1, SEQ_LEN, CONFIG["vocab_size"])
del test_logits
torch.cuda.empty_cache()
print("Forward pass passed.")

Parameters: 236,185,681
Forward pass passed.


In [5]:
def token_sequences(split):
    dataset = load_dataset(
        "roneneldan/TinyStories",
        split=split,
        streaming=True,
    )

    buffer = []
    position = 0
    chunk_size = SEQ_LEN + 1

    for example in dataset:
        text = example["text"]
        if not text.strip():
            continue

        ids = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            verbose=False,
        )["input_ids"]

        buffer.extend(ids)
        buffer.append(tokenizer.eos_token_id)

        while len(buffer) - position >= chunk_size:
            yield torch.tensor(
                buffer[position:position + chunk_size],
                dtype=torch.long,
            )
            position += chunk_size

        # Discard consumed tokens to keep memory bounded.
        if position:
            buffer = buffer[position:]
            position = 0


def batches(split):
    pending = []
    for sequence in token_sequences(split):
        pending.append(sequence)
        if len(pending) == BATCH_SIZE:
            yield torch.stack(pending)
            pending.clear()


def infinite_training_batches():
    while True:
        produced = False
        for batch in batches("train"):
            produced = True
            yield batch
        if not produced:
            raise RuntimeError("Dataset yielded no complete batches.")


# Keep a fixed, small validation sample on CPU.
validation_batches = []
for batch in batches("validation"):
    validation_batches.append(batch)
    if len(validation_batches) >= VALIDATION_BATCHES:
        break

assert validation_batches, "No validation batches were produced."
print("Validation batches:", len(validation_batches))

README.md: 0.00B [00:00, ?B/s]

Validation batches: 10


In [6]:
import json
import os

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.95),
    weight_decay=0.1,
)
scaler = torch.amp.GradScaler("cuda")

CHECKPOINT = SAVE_DIR / "training_latest.pt"
start_step = 0
history = []

if RESUME and CHECKPOINT.exists():
    # Only load trusted checkpoints created by your own notebook.
    checkpoint = torch.load(
        CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
    assert checkpoint["config"] == CONFIG, "Architecture mismatch."

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scaler.load_state_dict(checkpoint["scaler"])

    # Move restored optimizer tensors to the GPU.
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)

    start_step = checkpoint["step"]
    history = checkpoint.get("history", [])
    del checkpoint
    print("Resuming after optimizer step:", start_step)


@torch.no_grad()
def validate():
    model.eval()
    total_loss = 0.0

    for batch in validation_batches:
        batch = batch.to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(batch[:, :-1])

        loss = F.cross_entropy(
            logits.float().reshape(-1, CONFIG["vocab_size"]),
            batch[:, 1:].reshape(-1),
        )
        total_loss += loss.item()

    model.train()
    return total_loss / len(validation_batches)


def save_training_checkpoint(step):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "step": step,
        "config": CONFIG,
        "history": history,
        "dataset": "roneneldan/TinyStories",
        "tokenizer": "gpt2",
        "sequence_length": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
    }

    temporary = SAVE_DIR / "training_latest.tmp"
    torch.save(payload, temporary)
    os.replace(temporary, CHECKPOINT)

    print("Saved resumable checkpoint:", CHECKPOINT)


print(
    "Note: resuming restores training state, but starts the dataset stream "
    "again rather than restoring its exact position."
)

Note: resuming restores training state, but starts the dataset stream again rather than restoring its exact position.


In [11]:
best_validation_loss = min(
    (entry["validation_loss"] for entry in history),
    default=float("inf"),
)

In [7]:
train_iterator = infinite_training_batches()
model.train()
completed_step = start_step

for step in range(start_step, STEPS):
    # Warmup followed by cosine decay.
    warmup_steps = max(1, min(100, STEPS // 10))

    if step < warmup_steps:
        learning_rate = LR * (step + 1) / warmup_steps
    else:
        progress = (
            (step - warmup_steps)
            / max(1, STEPS - warmup_steps)
        )
        learning_rate = LR * (
            0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))
        )

    for group in optimizer.param_groups:
        group["lr"] = learning_rate

    optimizer.zero_grad(set_to_none=True)
    training_loss = 0.0

    for _ in range(GRAD_ACCUM):
        batch = next(train_iterator).to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(batch[:, :-1])

        loss = F.cross_entropy(
            logits.float().reshape(-1, CONFIG["vocab_size"]),
            batch[:, 1:].reshape(-1),
        )

        if not torch.isfinite(loss):
            raise RuntimeError(f"Nonfinite loss at step {step + 1}")

        training_loss += loss.item()
        scaler.scale(loss / GRAD_ACCUM).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

    completed_step = step + 1
    mean_loss = training_loss / GRAD_ACCUM

    if completed_step % 10 == 0:
        print(
            f"Step {completed_step}/{STEPS} "
            f"| train loss {mean_loss:.4f} "
            f"| LR {learning_rate:.2e}"
        )

    if completed_step % VALIDATE_EVERY == 0:
        validation_loss = validate()
        history.append({
            "step": completed_step,
            "train_loss": mean_loss,
            "validation_loss": validation_loss,
        })
        print(f"Validation loss: {validation_loss:.4f}")

    if completed_step % SAVE_EVERY == 0:
        save_training_checkpoint(completed_step)

save_training_checkpoint(completed_step)
print("Training finished.")

Step 10/100 | train loss 9.4377 | LR 3.00e-04
Step 20/100 | train loss 6.6846 | LR 2.93e-04
Step 30/100 | train loss 5.4318 | LR 2.71e-04
Step 40/100 | train loss 5.2684 | LR 2.37e-04
Step 50/100 | train loss 4.5692 | LR 1.93e-04
Step 60/100 | train loss 4.8942 | LR 1.46e-04
Step 70/100 | train loss 4.7632 | LR 1.02e-04
Step 80/100 | train loss 4.6478 | LR 6.47e-05
Step 90/100 | train loss 4.7170 | LR 3.98e-05
Step 100/100 | train loss 4.5150 | LR 3.01e-05
Validation loss: 4.4345
Saved resumable checkpoint: /content/drive/MyDrive/AttentionLens_RetNet/training_latest.pt
Saved resumable checkpoint: /content/drive/MyDrive/AttentionLens_RetNet/training_latest.pt
Training finished.


In [8]:
final_validation_loss = validate()

weights_path = SAVE_DIR / "retnet_128.pt"
torch.save(
    {name: tensor.detach().cpu()
     for name, tensor in model.state_dict().items()},
    weights_path,
)

metadata = {
    "architecture": "AttentionLens TorchScale RetNetModel",
    "config": CONFIG,
    "dataset": "roneneldan/TinyStories",
    "tokenizer": "gpt2",
    "optimizer_steps": completed_step,
    "sequence_length": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "validation_loss": final_validation_loss,
    "history": history,
}

metadata_path = SAVE_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2))

# Confirm the exported weights load strictly into the same architecture.
check_model = RetNetModel(**CONFIG)
state = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True,
)
check_model.load_state_dict(state, strict=True)
del check_model, state

print("Compatible checkpoint:", weights_path)
print("Training metadata:", metadata_path)
print("Final validation loss:", final_validation_loss)

Compatible checkpoint: /content/drive/MyDrive/AttentionLens_RetNet/retnet_128.pt
Training metadata: /content/drive/MyDrive/AttentionLens_RetNet/training_metadata.json
Final validation loss: 4.434524583816528
